# Retrieval-Augmented Video Dehazing

A complete implementation of the Retrieval-Augmented Video Dehazing pipeline.

**Pipeline Overview:**
1. **Ground-truth path:** Ground truth video → frame extraction → feature extraction → embeddings → stored in VectorDB
2. **Hazy path:** Input hazy video → encoder → feature extraction → hazy vectors → VectorDB query → vector dehazing → dehazed vectors → decoder → dehazed video output

**Steps implemented:**
- Step 1: Dataset D = {(Hᵢ, Gᵢ)}
- Step 2: Feature extraction via encoder E(·)
- Step 3: VectorDB storage V = {Zₕ, Zg}
- Step 4: VectorDB query via cosine similarity
- Step 5: Feature aggregation (average + attention)
- Step 6: Dehazing network g^(X) where X = [Zq, R]
- Step 7: Combined loss L = W1·L1 + W2·L2 + W3·L3
- Step 8: Reconstruction & video encoding

## 1. Install Dependencies

In [ ]:
!pip install -q faiss-cpu opencv-python-headless

## 2. Imports & Configuration

In [ ]:
import os
import glob
import math
import random
import numpy as np
from PIL import Image

import cv2
import faiss

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as T
import torchvision.models as models

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Configuration — all hyper-parameters & paths in one place
# ═══════════════════════════════════════════════════════════════

class Config:
    # ---- Data ----
    # Set these to your Kaggle dataset paths
    # Option A: paired video files
    hazy_video_dir   = '/kaggle/input/your-dataset/hazy_videos'
    clean_video_dir  = '/kaggle/input/your-dataset/clean_videos'
    # Option B: paired image folders (pre-extracted frames)
    hazy_frame_dir   = '/kaggle/input/your-dataset/hazy_frames'
    clean_frame_dir  = '/kaggle/input/your-dataset/clean_frames'
    use_video_input  = False   # True → extract frames from videos; False → use image folders

    output_dir       = '/kaggle/working/output'
    checkpoint_dir   = '/kaggle/working/checkpoints'

    # ---- Frames ----
    frame_h          = 256
    frame_w          = 256

    # ---- Encoder ----
    embedding_size   = 512    # Es — dimension of z

    # ---- VectorDB (FAISS) ----
    top_k            = 5      # how many neighbours to retrieve
    similarity       = 'cosine'   # 'cosine' or 'l2'

    # ---- Aggregation ----
    aggregation      = 'attention'  # 'average' or 'attention'

    # ---- Dehazer ----
    dehazer_hidden   = 1024

    # ---- Loss weights ----
    w1               = 1.0    # pixel loss weight
    w2               = 0.1    # perceptual loss weight
    w3               = 0.05   # contrastive retrieval loss weight
    temperature      = 0.07   # τ for contrastive loss

    # ---- Training ----
    batch_size       = 8
    num_epochs       = 50
    learning_rate    = 1e-4
    weight_decay     = 1e-5
    num_workers      = 2

    # ---- Inference ----
    output_fps       = 30

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.checkpoint_dir, exist_ok=True)
print('Config ready.')

## 3. Video I/O Utilities

Extract frames from videos and reconstruct videos from frames.

In [ ]:
def extract_frames(video_path, resize=(256, 256)):
    """Extract all frames from a video file.

    Args:
        video_path: Path to the input video.
        resize: (H, W) to resize each frame.

    Returns:
        List of PIL.Image frames.
    """
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # BGR → RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(frame).resize((resize[1], resize[0]), Image.BILINEAR)
        frames.append(img)
    cap.release()
    return frames


def frames_to_video(frames, output_path, fps=30):
    """Encode a list of numpy frames (H,W,3 uint8 BGR) into an MP4 video.

    Args:
        frames: List of numpy arrays (H, W, 3) in BGR uint8.
        output_path: Where to save the .mp4 file.
        fps: Frames per second.
    """
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    for f in frames:
        writer.write(f)
    writer.release()
    print(f'Video saved → {output_path}  ({len(frames)} frames, {fps} fps)')


def tensor_to_numpy_bgr(tensor):
    """Convert a (C,H,W) float tensor [0,1] to a (H,W,3) uint8 BGR numpy array."""
    img = tensor.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    img = (img * 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    return img


print('Video I/O utilities loaded.')

## 4. Dataset — D = {(Hᵢ, Gᵢ)}

Paired hazy / ground-truth frames. Supports loading from:
- Pre-extracted frame folders, **or**
- Video files (frames extracted on the fly).

In [ ]:
class HazyCleanDataset(Dataset):
    """Dataset D = {(Hᵢ, Gᵢ)} of paired hazy and clean frames."""

    def __init__(self, hazy_dir, clean_dir, transform=None, from_video=False):
        """
        Args:
            hazy_dir:  Directory of hazy frames (images) or videos.
            clean_dir: Directory of corresponding clean frames or videos.
            transform: torchvision transform applied to each frame.
            from_video: If True, extract frames from video files.
        """
        self.transform = transform or T.Compose([
            T.Resize((cfg.frame_h, cfg.frame_w)),
            T.ToTensor(),   # → [0, 1] float32
        ])

        if from_video:
            # Collect all video pairs and extract frames
            hazy_vids  = sorted(glob.glob(os.path.join(hazy_dir, '*')))
            clean_vids = sorted(glob.glob(os.path.join(clean_dir, '*')))
            assert len(hazy_vids) == len(clean_vids), \
                f'Mismatch: {len(hazy_vids)} hazy vs {len(clean_vids)} clean videos'

            self.hazy_frames  = []
            self.clean_frames = []
            for hv, cv_ in zip(hazy_vids, clean_vids):
                hf = extract_frames(hv, resize=(cfg.frame_h, cfg.frame_w))
                cf = extract_frames(cv_, resize=(cfg.frame_h, cfg.frame_w))
                min_len = min(len(hf), len(cf))
                self.hazy_frames.extend(hf[:min_len])
                self.clean_frames.extend(cf[:min_len])
        else:
            # Pre-extracted image frames
            exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp')
            hazy_paths  = []
            clean_paths = []
            for ext in exts:
                hazy_paths.extend(glob.glob(os.path.join(hazy_dir, ext)))
                clean_paths.extend(glob.glob(os.path.join(clean_dir, ext)))
            hazy_paths  = sorted(hazy_paths)
            clean_paths = sorted(clean_paths)
            assert len(hazy_paths) == len(clean_paths), \
                f'Mismatch: {len(hazy_paths)} hazy vs {len(clean_paths)} clean frames'

            self.hazy_frames  = [Image.open(p).convert('RGB') for p in hazy_paths]
            self.clean_frames = [Image.open(p).convert('RGB') for p in clean_paths]

        print(f'Dataset loaded: {len(self.hazy_frames)} paired frames.')

    def __len__(self):
        return len(self.hazy_frames)

    def __getitem__(self, idx):
        hazy  = self.transform(self.hazy_frames[idx])
        clean = self.transform(self.clean_frames[idx])
        return hazy, clean


print('Dataset class ready.')

## 5. Step 2 — Feature Encoder E(·)

Every frame is converted to an embedding: `z = E(frame)` where `z ∈ ℝ^{Es}`.

We use a **ResNet-18** backbone (pre-trained on ImageNet) with the final FC layer replaced by a linear projection to `embedding_size` dimensions.

In [ ]:
class FrameEncoder(nn.Module):
    """Feature encoder E(·).

    Maps an input frame (3, H, W) → embedding z ∈ ℝ^{Es}.
    Shared between hazy and clean paths (weight-tied).
    """

    def __init__(self, embedding_size=512):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Remove the final classification head
        self.features = nn.Sequential(*list(backbone.children())[:-1])  # → (B, 512, 1, 1)
        backbone_dim = 512  # ResNet-18 last conv output channels

        # Projection to desired embedding size
        self.projector = nn.Sequential(
            nn.Linear(backbone_dim, embedding_size),
            nn.ReLU(inplace=True),
            nn.Linear(embedding_size, embedding_size),
        )

    def forward(self, x):
        """x: (B, 3, H, W) → z: (B, Es)"""
        feat = self.features(x)           # (B, 512, 1, 1)
        feat = feat.flatten(start_dim=1)  # (B, 512)
        z = self.projector(feat)           # (B, Es)
        return z


# Quick test
_enc = FrameEncoder(cfg.embedding_size).to(DEVICE)
_dummy = torch.randn(2, 3, cfg.frame_h, cfg.frame_w).to(DEVICE)
_z = _enc(_dummy)
print(f'Encoder output shape: {_z.shape}  (expected: [2, {cfg.embedding_size}])')
del _enc, _dummy, _z

## 6. Step 3 & 4 — VectorDB (FAISS)

- **Step 3:** Store embeddings as V = {Zₕ, Zg}
- **Step 4:** Query — given a hazy embedding Zk = E(Hk), retrieve the top-k most similar clean features using cosine similarity or Euclidean distance.

In [ ]:
class VectorDB:
    """FAISS-backed vector store for paired hazy/clean embeddings.

    Stores V = {Zₕ, Zg}.  A hazy query retrieves the most similar
    *clean* (ground-truth) embeddings.
    """

    def __init__(self, embedding_size, similarity='cosine'):
        """
        Args:
            embedding_size: Dimension of each embedding vector.
            similarity: 'cosine' → inner product on L2-normalised vectors;
                        'l2' → Euclidean distance.
        """
        self.embedding_size = embedding_size
        self.similarity = similarity

        if similarity == 'cosine':
            # Inner product on L2-normalised vectors == cosine similarity
            self.index = faiss.IndexFlatIP(embedding_size)
        else:
            self.index = faiss.IndexFlatL2(embedding_size)

        # Parallel arrays — same ordering as FAISS index
        self.clean_embeddings = []   # Zg counterparts
        self.hazy_embeddings  = []   # Zh stored for reference

    def add(self, z_hazy, z_clean):
        """Add paired embeddings to the store.

        Args:
            z_hazy:  (N, Es) numpy float32 — hazy embeddings.
            z_clean: (N, Es) numpy float32 — corresponding clean embeddings.
        """
        z_hazy  = np.ascontiguousarray(z_hazy.astype(np.float32))
        z_clean = np.ascontiguousarray(z_clean.astype(np.float32))

        if self.similarity == 'cosine':
            faiss.normalize_L2(z_hazy)

        # We index the *hazy* embeddings so queries find similar hazy entries,
        # then return the paired *clean* embedding.
        self.index.add(z_hazy)
        self.hazy_embeddings.append(z_hazy)
        self.clean_embeddings.append(z_clean)

    def query(self, z_query, top_k=5):
        """Retrieve top-k most similar *clean* embeddings for a hazy query.

        Args:
            z_query: (B, Es) numpy float32 — query hazy embeddings.
            top_k:   Number of neighbours.

        Returns:
            retrieved: (B, top_k, Es) numpy float32 — clean embeddings.
        """
        z_query = np.ascontiguousarray(z_query.astype(np.float32))
        if self.similarity == 'cosine':
            faiss.normalize_L2(z_query)

        distances, indices = self.index.search(z_query, top_k)  # (B, top_k)

        # Gather all clean embeddings into one contiguous array
        all_clean = np.concatenate(self.clean_embeddings, axis=0)  # (Total, Es)

        batch_size = z_query.shape[0]
        retrieved = np.zeros((batch_size, top_k, self.embedding_size), dtype=np.float32)
        for i in range(batch_size):
            for j in range(top_k):
                idx = indices[i, j]
                if idx >= 0:  # FAISS returns -1 for missing entries
                    retrieved[i, j] = all_clean[idx]

        return retrieved

    @property
    def size(self):
        return self.index.ntotal

    def reset(self):
        self.index.reset()
        self.clean_embeddings.clear()
        self.hazy_embeddings.clear()


# Quick test
_vdb = VectorDB(cfg.embedding_size, cfg.similarity)
_zh = np.random.randn(10, cfg.embedding_size).astype(np.float32)
_zc = np.random.randn(10, cfg.embedding_size).astype(np.float32)
_vdb.add(_zh, _zc)
_q  = np.random.randn(2, cfg.embedding_size).astype(np.float32)
_ret = _vdb.query(_q, top_k=3)
print(f'VectorDB size: {_vdb.size}, query result shape: {_ret.shape}  (expected: [2, 3, {cfg.embedding_size}])')
del _vdb, _zh, _zc, _q, _ret

## 7. Step 5 — Feature Aggregation

After retrieval we get embeddings r₁, r₂, …, rₖ. We combine them into a single context embedding **R** via:
- **Simple average:** `R = (1/k) Σ rᵢ`
- **Attention-based:** learn attention weights conditioned on the query.

In [ ]:
class AverageAggregator(nn.Module):
    """Simple average over retrieved embeddings."""

    def forward(self, query, retrieved):
        """query: (B, Es), retrieved: (B, K, Es) → R: (B, Es)"""
        return retrieved.mean(dim=1)


class AttentionAggregator(nn.Module):
    """Attention-based aggregation conditioned on the hazy query.

    Computes scaled dot-product attention between the query and
    each retrieved embedding to produce a weighted sum.
    """

    def __init__(self, embedding_size):
        super().__init__()
        self.query_proj = nn.Linear(embedding_size, embedding_size)
        self.key_proj   = nn.Linear(embedding_size, embedding_size)
        self.scale = math.sqrt(embedding_size)

    def forward(self, query, retrieved):
        """query: (B, Es), retrieved: (B, K, Es) → R: (B, Es)"""
        Q = self.query_proj(query).unsqueeze(1)        # (B, 1, Es)
        K = self.key_proj(retrieved)                    # (B, K, Es)
        attn = torch.bmm(Q, K.transpose(1, 2)) / self.scale  # (B, 1, K)
        attn = F.softmax(attn, dim=-1)
        R = torch.bmm(attn, retrieved).squeeze(1)      # (B, Es)
        return R


def build_aggregator(method='attention', embedding_size=512):
    if method == 'average':
        return AverageAggregator()
    else:
        return AttentionAggregator(embedding_size)


print('Aggregators ready.')

## 8. Step 6 — Dehazing Network g^(X)

Concatenates hazy embedding Zq and context embedding R → X = [Zq, R].

The network learns to map from the combined embedding to the predicted clean embedding, which is then decoded back to pixel space via a CNN decoder.

In [ ]:
class DehazingNetwork(nn.Module):
    """Dehazing network: maps X = [Zq, R] → predicted clean embedding.

    Also includes a spatial decoder that converts embeddings back
    to pixel-space images.
    """

    def __init__(self, embedding_size=512, hidden_size=1024,
                 out_h=256, out_w=256):
        super().__init__()
        self.out_h = out_h
        self.out_w = out_w

        # ── Embedding-space fusion MLP ──
        # Input: [Zq, R] → 2 * embedding_size
        self.fusion = nn.Sequential(
            nn.Linear(2 * embedding_size, hidden_size),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, embedding_size),
        )

        # ── Spatial decoder: embedding → image ──
        # Project embedding to a small spatial feature map, then upsample.
        init_spatial = 8  # start at 8×8
        self.embed_to_spatial = nn.Linear(embedding_size, 256 * init_spatial * init_spatial)
        self.init_spatial = init_spatial

        self.decoder = nn.Sequential(
            # 256 × 8 × 8 → 128 × 16 × 16
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            # 128 × 16 × 16 → 64 × 32 × 32
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            # 64 × 32 × 32 → 32 × 64 × 64
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            # 32 × 64 × 64 → 16 × 128 × 128
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            # 16 × 128 × 128 → 3 × 256 × 256
            nn.ConvTranspose2d(16, 3, 4, stride=2, padding=1),
            nn.Sigmoid(),  # output in [0, 1]
        )

    def forward(self, z_hazy, context):
        """Forward pass.

        Args:
            z_hazy:  (B, Es) — hazy embedding Zq.
            context: (B, Es) — aggregated context R.

        Returns:
            pred_clean_embed: (B, Es) — predicted clean embedding.
            pred_image:       (B, 3, H, W) — decoded predicted clean image.
        """
        # Step 6: X = [Zq, R]
        x = torch.cat([z_hazy, context], dim=1)   # (B, 2*Es)
        pred_clean_embed = self.fusion(x)          # (B, Es)

        # Decode to image
        spatial = self.embed_to_spatial(pred_clean_embed)  # (B, 256*8*8)
        spatial = spatial.view(-1, 256, self.init_spatial, self.init_spatial)  # (B,256,8,8)
        pred_image = self.decoder(spatial)  # (B, 3, 256, 256)

        return pred_clean_embed, pred_image


# Quick test
_dh = DehazingNetwork(cfg.embedding_size, cfg.dehazer_hidden,
                      cfg.frame_h, cfg.frame_w).to(DEVICE)
_zh = torch.randn(2, cfg.embedding_size).to(DEVICE)
_ctx = torch.randn(2, cfg.embedding_size).to(DEVICE)
_embed, _img = _dh(_zh, _ctx)
print(f'Dehazer embed shape: {_embed.shape}, image shape: {_img.shape}')
del _dh, _zh, _ctx, _embed, _img

## 9. Step 7 — Loss Functions

| Loss | Formula |
|------|--------|
| **Pixel (L1)** | `(1/N) Σ \|gᵢ − ĝᵢ\|` |
| **Perceptual (L2)** | `\|ϕ(ĝ) − ϕ(g)\|` where ϕ = VGG-16 features |
| **Contrastive Retrieval (L3)** | `−log[ e^{cos(Zₕ,Zg)/τ} / Σ e^{cos(Zₕ,Zⱼ)/τ} ]` |

Total: `L = W1·L1 + W2·L2 + W3·L3`

In [ ]:
class VGGPerceptualLoss(nn.Module):
    """Perceptual loss using VGG-16 features.

    L2 = ||ϕ(ĝ) - ϕ(g)||  where ϕ extracts intermediate features.
    """

    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        # Use features up to relu3_3 (layer index 15)
        self.feature_extractor = nn.Sequential(*list(vgg.features)[:16]).eval()
        # Freeze VGG
        for p in self.feature_extractor.parameters():
            p.requires_grad = False

        # ImageNet normalisation
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred, target):
        """pred, target: (B, 3, H, W) in [0, 1]"""
        pred   = (pred   - self.mean) / self.std
        target = (target - self.mean) / self.std
        phi_pred   = self.feature_extractor(pred)
        phi_target = self.feature_extractor(target)
        return F.mse_loss(phi_pred, phi_target)


class ContrastiveRetrievalLoss(nn.Module):
    """Contrastive retrieval loss.

    L3 = -log[ e^{cos(Zₕ, Zg⁺) / τ} / Σⱼ e^{cos(Zₕ, Zgⱼ) / τ} ]

    The positive pair is (Zₕ, Zg) for the same frame.
    Negatives are all other clean embeddings in the batch.
    """

    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_hazy, z_clean):
        """z_hazy, z_clean: (B, Es) — paired embeddings."""
        # L2 normalise
        z_h = F.normalize(z_hazy, dim=1)
        z_c = F.normalize(z_clean, dim=1)

        # Similarity matrix (B, B)
        logits = torch.mm(z_h, z_c.t()) / self.temperature

        # Positive pairs along the diagonal
        labels = torch.arange(logits.size(0), device=logits.device)
        loss = F.cross_entropy(logits, labels)
        return loss


class CombinedLoss(nn.Module):
    """Total loss: L = W1·L1 + W2·L2 + W3·L3"""

    def __init__(self, w1=1.0, w2=0.1, w3=0.05, temperature=0.07):
        super().__init__()
        self.w1 = w1
        self.w2 = w2
        self.w3 = w3

        self.pixel_loss       = nn.L1Loss()                   # L1
        self.perceptual_loss  = VGGPerceptualLoss()           # L2
        self.contrastive_loss = ContrastiveRetrievalLoss(temperature)  # L3

    def forward(self, pred_image, target_image,
                z_hazy, z_clean_pred, z_clean_gt):
        """Compute combined loss.

        Args:
            pred_image:    (B, 3, H, W) predicted clean image.
            target_image:  (B, 3, H, W) ground truth clean image.
            z_hazy:        (B, Es) hazy embeddings.
            z_clean_pred:  (B, Es) predicted clean embeddings.
            z_clean_gt:    (B, Es) ground truth clean embeddings.

        Returns:
            total_loss, dict of individual losses.
        """
        l1 = self.pixel_loss(pred_image, target_image)
        l2 = self.perceptual_loss(pred_image, target_image)
        l3 = self.contrastive_loss(z_hazy, z_clean_gt)

        total = self.w1 * l1 + self.w2 * l2 + self.w3 * l3

        return total, {
            'pixel_L1': l1.item(),
            'perceptual_L2': l2.item(),
            'contrastive_L3': l3.item(),
            'total': total.item(),
        }


print('Loss functions ready.')

## 10. Training Pipeline

End-to-end training loop:
1. Populate VectorDB with encoder embeddings from the training set.
2. For each batch: encode hazy → query VectorDB → aggregate → dehaze → compute loss → backprop.

In [ ]:
def populate_vector_db(encoder, dataloader, vector_db, device):
    """Encode the full training set and populate the VectorDB.

    Step 2 & 3: Feature extraction + VectorDB storage.
    """
    vector_db.reset()
    encoder.eval()
    with torch.no_grad():
        for hazy, clean in dataloader:
            hazy  = hazy.to(device)
            clean = clean.to(device)
            z_h = encoder(hazy).cpu().numpy()    # Zh = E(ht)
            z_c = encoder(clean).cpu().numpy()   # Zg = E(gt)
            vector_db.add(z_h, z_c)
    print(f'VectorDB populated with {vector_db.size} entries.')


def train_one_epoch(encoder, dehazer, aggregator, criterion,
                    optimizer, dataloader, vector_db, device, cfg):
    """Train for one epoch."""
    encoder.train()
    dehazer.train()
    aggregator.train()

    epoch_losses = {'pixel_L1': 0, 'perceptual_L2': 0,
                    'contrastive_L3': 0, 'total': 0}
    n_batches = 0

    for hazy, clean in dataloader:
        hazy  = hazy.to(device)
        clean = clean.to(device)

        # Step 2: Encode
        z_hazy  = encoder(hazy)    # (B, Es)
        z_clean = encoder(clean)   # (B, Es)

        # Step 4: Query VectorDB with hazy embeddings
        z_hazy_np = z_hazy.detach().cpu().numpy()
        retrieved_np = vector_db.query(z_hazy_np, top_k=cfg.top_k)  # (B, K, Es)
        retrieved = torch.from_numpy(retrieved_np).to(device)

        # Step 5: Aggregate retrieved embeddings
        context = aggregator(z_hazy, retrieved)  # (B, Es)

        # Step 6: Dehaze — X = [Zq, R] → predicted clean
        pred_clean_embed, pred_image = dehazer(z_hazy, context)

        # Step 7: Compute loss
        loss, loss_dict = criterion(
            pred_image, clean,
            z_hazy, pred_clean_embed, z_clean
        )

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        for k in epoch_losses:
            epoch_losses[k] += loss_dict[k]
        n_batches += 1

    # Average
    for k in epoch_losses:
        epoch_losses[k] /= max(n_batches, 1)

    return epoch_losses


print('Training functions ready.')

In [ ]:
def train(cfg):
    """Full training loop."""

    # ── Dataset ──
    if cfg.use_video_input:
        dataset = HazyCleanDataset(cfg.hazy_video_dir, cfg.clean_video_dir,
                                   from_video=True)
    else:
        dataset = HazyCleanDataset(cfg.hazy_frame_dir, cfg.clean_frame_dir,
                                   from_video=False)

    dataloader = DataLoader(dataset, batch_size=cfg.batch_size,
                            shuffle=True, num_workers=cfg.num_workers,
                            drop_last=True)

    # ── Models ──
    encoder    = FrameEncoder(cfg.embedding_size).to(DEVICE)
    dehazer    = DehazingNetwork(cfg.embedding_size, cfg.dehazer_hidden,
                                cfg.frame_h, cfg.frame_w).to(DEVICE)
    aggregator = build_aggregator(cfg.aggregation, cfg.embedding_size).to(DEVICE)

    # ── Loss ──
    criterion = CombinedLoss(cfg.w1, cfg.w2, cfg.w3, cfg.temperature).to(DEVICE)

    # ── Optimizer ──
    params = (list(encoder.parameters()) +
              list(dehazer.parameters()) +
              list(aggregator.parameters()))
    optimizer = optim.Adam(params, lr=cfg.learning_rate,
                          weight_decay=cfg.weight_decay)

    # ── VectorDB ──
    vector_db = VectorDB(cfg.embedding_size, cfg.similarity)

    # ── Training loop ──
    best_loss = float('inf')

    for epoch in range(1, cfg.num_epochs + 1):
        # Re-populate VectorDB with current encoder weights every epoch
        populate_vector_db(encoder, dataloader, vector_db, DEVICE)

        # Train
        losses = train_one_epoch(
            encoder, dehazer, aggregator, criterion,
            optimizer, dataloader, vector_db, DEVICE, cfg
        )

        print(f'Epoch [{epoch}/{cfg.num_epochs}]  '
              f'Total: {losses["total"]:.4f}  '
              f'L1: {losses["pixel_L1"]:.4f}  '
              f'Percep: {losses["perceptual_L2"]:.4f}  '
              f'Contrastive: {losses["contrastive_L3"]:.4f}')

        # Save best checkpoint
        if losses['total'] < best_loss:
            best_loss = losses['total']
            checkpoint = {
                'epoch': epoch,
                'encoder':    encoder.state_dict(),
                'dehazer':    dehazer.state_dict(),
                'aggregator': aggregator.state_dict(),
                'optimizer':  optimizer.state_dict(),
                'loss':       best_loss,
            }
            path = os.path.join(cfg.checkpoint_dir, 'best_model.pth')
            torch.save(checkpoint, path)
            print(f'  ✓ Best model saved (loss={best_loss:.4f})')

    # Save final
    checkpoint = {
        'epoch': cfg.num_epochs,
        'encoder':    encoder.state_dict(),
        'dehazer':    dehazer.state_dict(),
        'aggregator': aggregator.state_dict(),
        'optimizer':  optimizer.state_dict(),
        'loss':       losses['total'],
    }
    path = os.path.join(cfg.checkpoint_dir, 'final_model.pth')
    torch.save(checkpoint, path)
    print(f'\nTraining complete. Final model saved to {path}')

    return encoder, dehazer, aggregator, vector_db

## 11. Step 8 — Inference & Reconstruction

Load a trained model, process an input hazy video frame-by-frame:
1. Encode each hazy frame → query VectorDB → aggregate → dehaze.
2. Collect predicted frames ĥ = {ĥ₁, ĥ₂, …, ĥT}.
3. Reconstruct output video Ĝ = {ĝ₁, ĝ₂, …, ĝT} using video encoding.

In [ ]:
@torch.no_grad()
def inference(hazy_video_path, encoder, dehazer, aggregator,
              vector_db, cfg, output_path=None):
    """Run inference on a hazy video and produce a dehazed output video.

    Step 8: Reconstruction.

    Args:
        hazy_video_path: Path to input hazy video file.
        encoder, dehazer, aggregator: Trained models.
        vector_db: Populated VectorDB.
        cfg: Configuration.
        output_path: Where to save the dehazed video.

    Returns:
        List of dehazed frame tensors.
    """
    encoder.eval()
    dehazer.eval()
    aggregator.eval()

    if output_path is None:
        output_path = os.path.join(cfg.output_dir, 'dehazed_output.mp4')

    # Extract frames from hazy video
    transform = T.Compose([
        T.Resize((cfg.frame_h, cfg.frame_w)),
        T.ToTensor(),
    ])

    hazy_frames = extract_frames(hazy_video_path,
                                  resize=(cfg.frame_h, cfg.frame_w))
    print(f'Processing {len(hazy_frames)} frames ...')

    dehazed_frames_bgr = []
    dehazed_tensors    = []

    for i, frame_pil in enumerate(hazy_frames):
        frame_tensor = transform(frame_pil).unsqueeze(0).to(DEVICE)  # (1,3,H,W)

        # Step 2: Encode
        z_hazy = encoder(frame_tensor)  # (1, Es)

        # Step 4: Query VectorDB
        z_np = z_hazy.cpu().numpy()
        retrieved_np = vector_db.query(z_np, top_k=cfg.top_k)  # (1,K,Es)
        retrieved = torch.from_numpy(retrieved_np).to(DEVICE)

        # Step 5: Aggregate
        context = aggregator(z_hazy, retrieved)  # (1, Es)

        # Step 6: Dehaze
        _, pred_image = dehazer(z_hazy, context)  # (1,3,H,W)

        dehazed_tensors.append(pred_image.squeeze(0).cpu())
        dehazed_frames_bgr.append(tensor_to_numpy_bgr(pred_image.squeeze(0)))

        if (i + 1) % 50 == 0:
            print(f'  Processed {i + 1}/{len(hazy_frames)} frames')

    # Step 8: Encode back to video
    frames_to_video(dehazed_frames_bgr, output_path, fps=cfg.output_fps)

    return dehazed_tensors


def inference_from_frames(hazy_frame_dir, encoder, dehazer, aggregator,
                          vector_db, cfg, output_path=None):
    """Run inference on a directory of hazy frame images.

    Same pipeline as above but reads from image files instead of a video.
    """
    encoder.eval()
    dehazer.eval()
    aggregator.eval()

    if output_path is None:
        output_path = os.path.join(cfg.output_dir, 'dehazed_output.mp4')

    transform = T.Compose([
        T.Resize((cfg.frame_h, cfg.frame_w)),
        T.ToTensor(),
    ])

    exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp')
    frame_paths = []
    for ext in exts:
        frame_paths.extend(glob.glob(os.path.join(hazy_frame_dir, ext)))
    frame_paths = sorted(frame_paths)
    print(f'Processing {len(frame_paths)} frames from {hazy_frame_dir}')

    dehazed_frames_bgr = []

    for i, fp in enumerate(frame_paths):
        img = Image.open(fp).convert('RGB')
        frame_tensor = transform(img).unsqueeze(0).to(DEVICE)

        z_hazy = encoder(frame_tensor)
        z_np = z_hazy.cpu().numpy()
        retrieved_np = vector_db.query(z_np, top_k=cfg.top_k)
        retrieved = torch.from_numpy(retrieved_np).to(DEVICE)
        context = aggregator(z_hazy, retrieved)
        _, pred_image = dehazer(z_hazy, context)

        dehazed_frames_bgr.append(tensor_to_numpy_bgr(pred_image.squeeze(0)))

        if (i + 1) % 50 == 0:
            print(f'  Processed {i + 1}/{len(frame_paths)} frames')

    frames_to_video(dehazed_frames_bgr, output_path, fps=cfg.output_fps)
    print('Inference complete.')


print('Inference functions ready.')

## 12. Load Checkpoint Utility

In [ ]:
def load_checkpoint(checkpoint_path, cfg):
    """Load a saved checkpoint and return models + vector_db."""
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)

    encoder = FrameEncoder(cfg.embedding_size).to(DEVICE)
    encoder.load_state_dict(ckpt['encoder'])

    dehazer = DehazingNetwork(cfg.embedding_size, cfg.dehazer_hidden,
                              cfg.frame_h, cfg.frame_w).to(DEVICE)
    dehazer.load_state_dict(ckpt['dehazer'])

    aggregator = build_aggregator(cfg.aggregation, cfg.embedding_size).to(DEVICE)
    aggregator.load_state_dict(ckpt['aggregator'])

    print(f'Checkpoint loaded from {checkpoint_path} (epoch {ckpt["epoch"]}, loss {ckpt["loss"]:.4f})')
    return encoder, dehazer, aggregator


print('Checkpoint loader ready.')

## 13. Run Training

**Before running:** Update `Config` paths above to point to your Kaggle dataset.

Expected directory structure:
```
/kaggle/input/your-dataset/
├── hazy_frames/       # Hazy frame images (001.png, 002.png, ...)
└── clean_frames/      # Corresponding clean frames (001.png, 002.png, ...)
```
or
```
/kaggle/input/your-dataset/
├── hazy_videos/       # Hazy video files
└── clean_videos/      # Corresponding clean video files
```

In [ ]:
# ══════════════════════════════════════════════════════════════
#  TRAIN
#  Uncomment the lines below to run training.
#  Make sure to update cfg paths first!
# ══════════════════════════════════════════════════════════════

# encoder, dehazer, aggregator, vector_db = train(cfg)

## 14. Run Inference

Load a trained checkpoint and process a new hazy video.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  INFERENCE
#  Uncomment the lines below to run inference.
# ══════════════════════════════════════════════════════════════

# # Load trained model
# ckpt_path = os.path.join(cfg.checkpoint_dir, 'best_model.pth')
# encoder, dehazer, aggregator = load_checkpoint(ckpt_path, cfg)

# # Re-populate VectorDB from training data
# dataset = HazyCleanDataset(cfg.hazy_frame_dir, cfg.clean_frame_dir, from_video=False)
# dataloader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
# vector_db = VectorDB(cfg.embedding_size, cfg.similarity)
# populate_vector_db(encoder, dataloader, vector_db, DEVICE)

# # Option A: Inference on a video file
# # dehazed = inference('/path/to/hazy_video.mp4',
# #                     encoder, dehazer, aggregator, vector_db, cfg)

# # Option B: Inference on a folder of hazy frames
# # inference_from_frames('/path/to/hazy_test_frames',
# #                       encoder, dehazer, aggregator, vector_db, cfg)

## 15. Visualise Results

In [ ]:
import matplotlib.pyplot as plt

def visualise_comparison(hazy_frames, dehazed_frames, clean_frames=None,
                         num_samples=5):
    """Display hazy → dehazed (→ clean) comparisons."""
    n = min(num_samples, len(hazy_frames))
    cols = 3 if clean_frames is not None else 2
    fig, axes = plt.subplots(n, cols, figsize=(4 * cols, 4 * n))
    if n == 1:
        axes = [axes]

    for i in range(n):
        # Hazy
        hazy_np = hazy_frames[i].permute(1, 2, 0).numpy()
        axes[i][0].imshow(hazy_np.clip(0, 1))
        axes[i][0].set_title('Hazy Input')
        axes[i][0].axis('off')

        # Dehazed
        dh_np = dehazed_frames[i].permute(1, 2, 0).numpy()
        axes[i][1].imshow(dh_np.clip(0, 1))
        axes[i][1].set_title('Dehazed Output')
        axes[i][1].axis('off')

        # Ground truth
        if clean_frames is not None:
            gt_np = clean_frames[i].permute(1, 2, 0).numpy()
            axes[i][2].imshow(gt_np.clip(0, 1))
            axes[i][2].set_title('Ground Truth')
            axes[i][2].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.output_dir, 'comparison.png'), dpi=150)
    plt.show()
    print(f'Comparison saved → {cfg.output_dir}/comparison.png')


# Uncomment to visualise after inference:
# visualise_comparison(hazy_tensors, dehazed_tensors, clean_tensors, num_samples=5)

## Architecture Summary

```
┌─────────────────────────────────────────────────────────────────────┐
│                  RETRIEVAL-AUGMENTED VIDEO DEHAZING                │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  Ground Truth Path (offline, populates VectorDB):                   │
│    Clean Video → Frames → Encoder E(·) → Zg → VectorDB             │
│    Hazy  Video → Frames → Encoder E(·) → Zh → VectorDB             │
│                                                                     │
│  Dehazing Path (online):                                            │
│    Hazy Frame Hk                                                    │
│        │                                                            │
│        ▼                                                            │
│    Encoder E(·) → Zq (hazy embedding)                               │
│        │                                                            │
│        ├──→ Query VectorDB (cosine sim) → r₁,r₂,...,rₖ              │
│        │                    │                                       │
│        │              Aggregator (avg / attention) → R              │
│        │                    │                                       │
│        └──→ Concat [Zq, R] = X                                     │
│                    │                                                │
│              Dehazing Network g^(X)                                  │
│                    │                                                │
│              ┌─────┴──────┐                                         │
│              │            │                                         │
│        Clean Embed    Decoder → Predicted Clean Frame ĝ             │
│              │            │                                         │
│              └────────────┘                                         │
│                    │                                                │
│              Loss: L = W1·L1 + W2·L2 + W3·L3                       │
│                                                                     │
│  Reconstruction: ĝ₁,ĝ₂,...,ĝT → Video Encoder → Dehazed Video     │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```